In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# --- Configuration ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
proxy_batch = 256
epochs = 30
warmup_epochs = 5
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# ============================================================
# Dataset Loading
# ============================================================
class StandardizedDataset(Dataset):
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        if isinstance(y, np.ndarray):
            y = int(y.item()) if y.size == 1 else int(y[0])
        elif torch.is_tensor(y):
            y = int(y.item())
        else:
            y = int(y)
        return x, y

def get_dataset(name):
    if name == 'cifar10':
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        train = datasets.CIFAR10(root='./data', train=True, transform=tfm, download=True)
        test = datasets.CIFAR10(root='./data', train=False, transform=tfm, download=True)
        return StandardizedDataset(train), StandardizedDataset(test), 3, 10

    elif name == 'bloodmnist':
        from medmnist import BloodMNIST
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        train = BloodMNIST(split='train', transform=tfm, download=True)
        test = BloodMNIST(split='test', transform=tfm, download=True)
        return StandardizedDataset(train), StandardizedDataset(test), 3, 8
    else:
        raise ValueError(f"Unknown dataset: {name}")

# ============================================================
# Model & Feature Extraction
# ============================================================
def get_model(in_channels, num_classes):
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

class EmbeddingExtractor:
    def __init__(self, model):
        self.features = None
        model.avgpool.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.features = out.flatten(1).detach()

# ============================================================
# Proxy Scoring
# ============================================================
def compute_proxy_signals(train_dataset, in_channels, num_classes, n_train):
    proxy = get_model(in_channels, num_classes)
    extractor = EmbeddingExtractor(proxy)
    opt = optim.Adam(proxy.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(reduction='none')
    loader = DataLoader(train_dataset, batch_size=proxy_batch, shuffle=False)

    proxy.train()
    for epoch in range(warmup_epochs):
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            out = proxy(x)
            loss = criterion(out, y)
            opt.zero_grad()
            loss.mean().backward()
            opt.step()
        print(f"    Proxy warm-up epoch {epoch + 1}/{warmup_epochs} done")

    proxy.eval()
    embeddings = np.zeros((n_train, 512), dtype=np.float32)
    el2n_scores = np.zeros(n_train, dtype=np.float32)
    eye = torch.eye(num_classes, device=device)

    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            out = proxy(x)
            emb = extractor.features
            probs = torch.softmax(out, dim=1)
            err = probs - eye[y]

            start = i * proxy_batch
            end = start + x.size(0)
            embeddings[start:end] = emb.cpu().numpy()
            el2n_scores[start:end] = torch.norm(err, dim=1).cpu().numpy()

    return embeddings, el2n_scores

# ============================================================
# HYBRID PPR-Coreset Selection
# ============================================================
def compute_hybrid_ppr_and_select(embeddings, el2n_scores, num_samples, k_neighbors=10, alpha=0.15, hybrid_ratio=0.5):
    n = embeddings.shape[0]
    if num_samples >= n:
        return np.arange(n)

    # 1. Budget Split
    num_random = int(num_samples * hybrid_ratio)
    num_ppr = num_samples - num_random

    # 2. Random Skeleton Selection
    random_indices = np.random.choice(n, num_random, replace=False)
    
    if num_ppr == 0:
        return random_indices

    # 3. PPR Graph Build
    nn_model = NearestNeighbors(n_neighbors=k_neighbors, metric='cosine').fit(embeddings)
    adj = nn_model.kneighbors_graph(embeddings, mode='connectivity')

    row_sums = np.array(adj.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    inv_row_sums = 1.0 / row_sums
    P = adj.multiply(inv_row_sums[:, None])

    v = el2n_scores / (el2n_scores.sum() + 1e-8)
    pi = v.copy()
    for _ in range(20):
        pi = (1 - alpha) * v + alpha * (P.T @ pi)

    # 4. Greedy Selection with gentler penalty (0.85)
    selected_ppr = []
    current_gain = pi.copy()
    
    # FORBID the random indices from being picked again
    current_gain[random_indices] = -np.inf 
    adj_csr = adj.tocsr()

    for _ in range(num_ppr):
        idx = int(np.argmax(current_gain))
        selected_ppr.append(idx)
        neighbors = adj_csr[idx].indices
        valid_neighbors = neighbors[current_gain[neighbors] != -np.inf]
        if len(valid_neighbors) > 0:
            current_gain[valid_neighbors] *= 0.85 # Gentler Submodular Penalty
        current_gain[idx] = -np.inf

    # 5. Combine and return
    return np.concatenate([random_indices, np.array(selected_ppr)]).astype(int)

# ============================================================
# Training & Evaluation
# ============================================================
def train_and_eval(train_dataset, test_dataset, indices, in_channels, num_classes):
    train_loader = DataLoader(Subset(train_dataset, indices), batch_size=batch_size, shuffle=True)
    model = get_model(in_channels, num_classes)
    opt = optim.Adam(model.parameters(), lr=0.001)
    ce = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            opt.zero_grad()
            ce(model(x.to(device)), y.to(device)).backward()
            opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in DataLoader(test_dataset, batch_size=512):
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(test_dataset)

# ============================================================
# Main Loop - PART 1
# ============================================================
results = []
datasets_names = ['bloodmnist', 'cifar10']

for ds_name in datasets_names:
    print(f"\n=== Dataset: {ds_name} ===")
    train_dataset, test_dataset, in_channels, num_classes = get_dataset(ds_name)
    n_train = len(train_dataset)
    print(f"  n_train={n_train}, in_channels={in_channels}, num_classes={num_classes}")

    print("  Computing proxy signals (embeddings & EL2N)...")
    embeddings, el2n_scores = compute_proxy_signals(
        train_dataset, in_channels, num_classes, n_train
    )

    for pct in [100, 80, 50, 25, 5]:
        num_samples = int(n_train * (pct / 100))
        
        # Run HYBRID PPR Selection
        indices = compute_hybrid_ppr_and_select(embeddings, el2n_scores, num_samples)
        
        # Train and Evaluate
        acc = train_and_eval(train_dataset, test_dataset, indices, in_channels, num_classes)

        results.append({
            'Dataset': ds_name, 
            'Method': 'Hybrid-PPR', 
            'Percentage': pct,
            'NumSamples': len(indices), 
            'TestAcc': acc
        })
        
        print(f"  [{ds_name}] Method: Hybrid-PPR | Pct: {pct:3d}% | "
              f"N: {len(indices):6d} | Acc: {acc:.4f}")

    pd.DataFrame(results).to_csv('hybrid_ppr_results_part1.csv', index=False)

print("\nExperiment complete. Results saved to hybrid_ppr_results_part1.csv")


=== Dataset: bloodmnist ===


100%|██████████| 35.5M/35.5M [00:18<00:00, 1.95MB/s]


  n_train=11959, in_channels=3, num_classes=8
  Computing proxy signals (embeddings & EL2N)...
    Proxy warm-up epoch 1/5 done
    Proxy warm-up epoch 2/5 done
    Proxy warm-up epoch 3/5 done
    Proxy warm-up epoch 4/5 done
    Proxy warm-up epoch 5/5 done
  [bloodmnist] Method: Hybrid-PPR | Pct: 100% | N:  11959 | Acc: 0.9211
  [bloodmnist] Method: Hybrid-PPR | Pct:  80% | N:   9567 | Acc: 0.9266
  [bloodmnist] Method: Hybrid-PPR | Pct:  50% | N:   5979 | Acc: 0.9228
  [bloodmnist] Method: Hybrid-PPR | Pct:  25% | N:   2989 | Acc: 0.8910
  [bloodmnist] Method: Hybrid-PPR | Pct:   5% | N:    597 | Acc: 0.8030

=== Dataset: cifar10 ===


100%|██████████| 170M/170M [21:00<00:00, 135kB/s] 


  n_train=50000, in_channels=3, num_classes=10
  Computing proxy signals (embeddings & EL2N)...
    Proxy warm-up epoch 1/5 done
    Proxy warm-up epoch 2/5 done
    Proxy warm-up epoch 3/5 done
    Proxy warm-up epoch 4/5 done
    Proxy warm-up epoch 5/5 done
  [cifar10] Method: Hybrid-PPR | Pct: 100% | N:  50000 | Acc: 0.8367
  [cifar10] Method: Hybrid-PPR | Pct:  80% | N:  40000 | Acc: 0.8209
  [cifar10] Method: Hybrid-PPR | Pct:  50% | N:  25000 | Acc: 0.7815
  [cifar10] Method: Hybrid-PPR | Pct:  25% | N:  12500 | Acc: 0.6712
  [cifar10] Method: Hybrid-PPR | Pct:   5% | N:   2500 | Acc: 0.3790

Experiment complete. Results saved to hybrid_ppr_results_part1.csv
